# Fermi-Hubbard Model via Majorana Propagation

This notebook demonstrates how to simulate the real-time quantum dynamics of the **1D Fermi-Hubbard model** using the **Majorana Propagation (MP) simulator**. The goal is to track how the spin-up occupancy at a given site evolves in time starting from a simple, unentangled initial state.

This 60-site example comes from the following paper, using the same model parameters, the same half-filled Néel initial state, and the spin-up occupancy at the central site as the observable.
The expected runtime for the whole MP-based simulation in this notebook is around 10-15 seconds.

> G. S. Hartnett, K. S. Najafi, A. Khindanov, H. Liao, M. Schutzman, M. R. Hush, M. J. Biercuk, Y. Baum,  
> *Fast, accurate, high-resolution simulation of large-scale Fermi-Hubbard models on a digital quantum processor*,  
> arXiv:2605.04025 (2026). https://arxiv.org/abs/2605.04025

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np

from monoprop import MonomialPropagator
from monoprop.fermi_data import FermiCircuit, FermiEvGate, FermiOperator

## 1. Define the model
The 1D Fermi-Hubbard Hamiltonian is:

$$H = -t \sum_{i,\sigma} \left(c^\dagger_{i,\sigma}\, c_{i+1,\sigma} + \text{h.c.}\right) + U \sum_i n_{i,\uparrow} n_{i,\downarrow} - \mu \sum_{i,\sigma} n_{i,\sigma}$$

The fermionic modes are indexed by site with spin states interleaved: at each site $i$, spin-up occupies mode $2i$ and spin-down occupies mode $2i+1$. The function below returns the ordered list of local terms for the **first-order Trotter decomposition** — hopping bonds first, then on-site interactions, then chemical potential terms.

In [ ]:
def mode(site, spin):
    """Return the interleaved mode index for a given site and spin (even index for spin-up, odd for spin-down)."""
    return 2 * site if spin == "up" else 2 * site + 1


def hubbard_fermion_terms(num_sites, hopping, interaction, chemical_potential):
    """Return the ordered list of local FermionOperator terms for the first-order Trotter decomposition of the 1D Hubbard model."""
    terms = []

    # nearest-neighbour hopping (both spin species)
    for site in range(num_sites - 1):
        for spin in ("up", "down"):
            left, right = mode(site, spin), mode(site + 1, spin)
            op_terms = [((left, "+"), (right, "-")), ((right, "+"), (left, "-"))]
            terms.append(
                FermiOperator(terms=op_terms, coefficients=[-hopping, -hopping])
            )

    # on-site Hubbard interaction
    for site in range(num_sites):
        up, down = mode(site, "up"), mode(site, "down")
        terms.append(
            FermiOperator(
                terms=[((up, "+"), (up, "-"), (down, "+"), (down, "-"))],
                coefficients=[interaction],
            )
        )

    # chemical potential (one term per spin-orbital)
    for site in range(num_sites):
        for spin in ("up", "down"):
            m = mode(site, spin)
            terms.append(
                FermiOperator(
                    terms=[((m, "+"), (m, "-"))],
                    coefficients=[-chemical_potential],
                )
            )

    return terms

Here are the key physical parameters for the model and computational parameters for the Trotter evolution:

- `num_sites` — number of lattice sites $N$ ($2N$ fermionic modes total).
- `hopping` — hopping amplitude $t = 1$ (energy scale).
- `interaction` — on-site interaction $U = -2$ (attractive).
- `chemical_potential` — set to 0 for half filling.
- `trotter_dt` — Trotter time step $\Delta t$.
- `trotter_steps` — number of Trotter steps.

In [ ]:
# physical parameters
num_sites = 60
hopping = 1.0
interaction = -2.0
chemical_potential = 0.0

# time-evolution parameters
trotter_dt = 0.2
trotter_steps = 30

# one mode per spin-orbital
num_qubits = 2 * num_sites

    "Each local Hamiltonian term is converted to Majorana generators and packed into four parallel arrays required by `propagate`: Majorana index tuples (`majoranas`), real prefactors (`gen_coeffs`), a group index mapping each generator to its local term (`param_inds`), and the shared time step per group (`parameters`). The real prefactor is obtained by factoring out the imaginary phase $(-i)^{w(w-1)/2}$ from each complex Majorana coefficient, where $w$ is the monomial weight."

In [ ]:
def build_trotter_layer(
    num_sites, hopping, interaction, chemical_potential, trotter_dt
):
    """Convert each local Hubbard term into Majorana generators and return them as four parallel arrays ready for the MP simulator."""
    ferm_ops = hubbard_fermion_terms(
        num_sites, hopping, interaction, chemical_potential
    )
    gates = [FermiEvGate(generator=term, parameter=trotter_dt) for term in ferm_ops]
    return FermiCircuit(initial_state=[], gates=gates)


fermi_circuit = build_trotter_layer(
    num_sites, hopping, interaction, chemical_potential, trotter_dt
)

We start with the half-filled Néel state as in the original paper. With `start_spin="down"`, even sites begin spin-down occupied and odd sites spin-up occupied; the function below returns the corresponding list of occupied mode indices.

In [ ]:
def neel_occupied_modes(num_sites, start_spin="up"):
    """Return the list of occupied mode indices for the half-filled Néel state, alternating spin between even and odd sites."""
    other = "down" if start_spin == "up" else "up"
    return [
        mode(site, start_spin if site % 2 == 0 else other) for site in range(num_sites)
    ]


occupied = neel_occupied_modes(num_sites, start_spin="down")
fermi_circuit.initial_state = occupied

We measure the spin-up occupancy observable $\langle n_{j,\uparrow} \rangle$ at the central site $j = \lfloor N/2 \rfloor$. In the Majorana basis:

$$n_{j,\uparrow} = \frac{1}{2}\mathbf{I} - \frac{i}{2}\,\gamma_{2j}\,\gamma_{2j+1}$$

The MP simulator tracks the monomial part; the constant shift $+\tfrac{1}{2}$ is accounted for analytically.

In [ ]:
def number_operator_majorana(site, spin):
    """Return the Majorana-basis representation of the number operator n_{site, spin}."""
    m = mode(site, spin)
    return FermiOperator(
        terms=[((m, "+"), (m, "-"))], coefficients=[1.0], num_modes=num_qubits
    )


obs_site = num_sites // 2
obs_spin = "up"
observable = number_operator_majorana(obs_site, obs_spin)

## 3. Run the simulation

    "`MonomialPropagator` maintains the observable as a sum of weighted Majorana monomials. Each call to `propagate` applies one Trotter step, conjugating each monomial by the local generators and generating higher-weight monomials through operator spreading.\n",

After each step, `simulator.expectation_value()` returns the observable $\langle n_{j,\uparrow} \rangle$ and `simulator.size()` the number of active monomials.

In [ ]:
simulator = MonomialPropagator(
    initial_operator=observable,
    quantum_circuit=fermi_circuit,
    cutoff=4,  # maximum Majorana monomial length kept
    cutoff_type="length",
    lower_atol=1e-4,
)

# check the initial state has 0 expectation value
print("Initial expectation value =", simulator.expectation_value())

In [ ]:
times = np.arange(trotter_steps + 1) * trotter_dt
values = np.empty(trotter_steps + 1)
term_counts = np.empty(trotter_steps + 1, dtype=int)

values[0] = simulator.expectation_value()
term_counts[0] = simulator.size()

for step in range(trotter_steps):
    simulator.propagate(evolve_with_coeffs=True)
    values[step + 1] = simulator.expectation_value()
    term_counts[step + 1] = simulator.size()
    print(
        f"  step {step + 1:2d}/{trotter_steps}  "
        f"t={times[step + 1]:.1f}  "
        f"<n>={values[step + 1]:.4f}  "
        f"terms={term_counts[step + 1]:,}"
    )

## 4. Results

The observable starts at zero (the central site is spin-down occupied in the Néel state), rises as hopping spreads spin-up electrons across the chain, and oscillates around the thermal value of $0.5$ (dashed line). The right panel shows the number of active Majorana monomials, which grows with time until truncation saturates the expansion.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

# left: expectation value vs time
ax = axes[0]
ax.plot(
    times,
    values,
    color="#33766f",
    linewidth=2,
    marker="s",
    markersize=6,
    markerfacecolor="none",
    markeredgecolor="#33766f",
    label="Majorana Propagation",
)
ax.axhline(0.5, color="gray", linestyle="--", linewidth=1, label="thermal")
ax.set_xlabel("Time $t$", fontsize=12)
ax.set_ylabel(rf"$\langle n_{{{obs_site},\uparrow}} \rangle$", fontsize=12)
ax.set_title(
    f"{num_sites}-site Hubbard model  ($U={interaction}$, $t={hopping}$)", fontsize=11
)
ax.set_xlim(0, times[-1])
ax.set_ylim(-0.05, 0.75)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, fontsize=9)

# right: active monomial count (proxy for computational cost)
ax2 = axes[1]
ax2.plot(
    times,
    term_counts,
    color="#7d1cb1",
    linewidth=2,
    marker="o",
    markersize=5,
    markerfacecolor="none",
)
ax2.set_xlabel("Time $t$", fontsize=12)
ax2.set_ylabel("Active Majorana monomials", fontsize=12)
ax2.set_title("Number of terms vs time", fontsize=11)
ax2.set_xlim(0, times[-1])
ax2.spines[["top", "right"]].set_visible(False)

plt.show()